In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

In [2]:
# ==========================
# CONECTAR GOOGLE DRIVE
# ==========================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ==========================
# CONFIGURACIÓN
# ==========================

DATASET_PATH = "/content/drive/MyDrive/dataset"
IMG_SIZE = 320
BATCH_SIZE = 8
EPOCHS = 10
NUM_CLASSES = 3

In [4]:
# ==========================
# VERIFICAR DATASET
# ==========================

print("Carpetas encontradas en dataset:")
print(os.listdir(DATASET_PATH))

Carpetas encontradas en dataset:
['organico', 'no_reciclables', 'reciclable']


In [5]:
# ==========================
# PREPROCESAMIENTO
# ==========================

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

# Datos de entrenamiento

train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Datos de validación

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 235 images belonging to 3 classes.
Found 57 images belonging to 3 classes.


In [6]:
# ==========================
# MODELO BASE
# ==========================

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

/tmp/ipykernel_470/1802159343.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
# ==========================
# MODELO FINAL
# ==========================

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

In [8]:
# ==========================
# COMPILACIÓN
# ==========================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 10, 10, 1280)   │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,586,691 (9.87 MB)

 Trainable params: 328,707 (1.25 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
# ==========================
# ENTRENAMIENTO
# ==========================

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 69s 2s/step - accuracy: 0.6426 - loss: 0.9758 - val_accuracy: 0.9298 - val_loss: 0.2185
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.9623 - loss: 0.1245 - val_accuracy: 0.8772 - val_loss: 0.3042
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.9840 - loss: 0.0819 - val_accuracy: 0.9298 - val_loss: 0.1522
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.9737 - loss: 0.0670 - val_accuracy: 0.9825 - val_loss: 0.1253
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9877 - loss: 0.0332 - val_accuracy: 0.9649 - val_loss: 0.1140
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.9991 - loss: 0.0165 - val_accuracy: 0.9649 - val_loss: 0.0642
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 1.0000 - loss: 0.0077 - val_accuracy: 0.8772 - val_loss: 0.3199
Epoch 8/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.9592 - loss: 0.0786 - val_accuracy: 0.9474 - val_loss:

In [10]:
# ==========================
# GUARDAR MODELO
# ==========================

model.save("/content/drive/MyDrive/modelo_basura_reentrenado.h5")

print("✅ Modelo guardado en Google Drive.")

✅ Modelo guardado en Google Drive.
